In [1]:
# ========== 导入：Groq 版 Day1 网页摘要工具箱 ==========

# 标准库 os：读环境变量（例如 GROQ_API_KEY）
import os
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 同目录 scraper：按 URL 抓取网页正文
from scraper import fetch_website_contents
# Markdown / display：在笔记本里漂亮展示模型输出
from IPython.display import Markdown, display
# Groq 官方 SDK：OpenAI 兼容的 chat.completions 接口，但跑在 Groq 加速推理上
from groq import Groq

# 若本格报错，请转到 troubleshooting notebook！


In [2]:
# ========== 从 .env 加载环境变量并检查 GROQ_API_KEY ==========

# override=True：.env 覆盖进程里已有的同名变量
load_dotenv(override=True)
# 取出 Groq 密钥
api_key = os.getenv('GROQ_API_KEY')

# ========== 检查 Key（打印文案保留英文，方便对照 troubleshooting） ==========

# 没读到密钥
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 首尾有空格/制表符
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
# ========== 预览：构造发给 Groq 的第一条 user 消息 ==========
# 有问题请看 Troubleshooting notebook。

# 自然语言问候（保留英文：prompt 内容会影响模型回复）
message = "Hello, Groq! This is my first ever message to you! Hi!"

# OpenAI 兼容的 messages 列表：此处只有一条 user
messages = [{"role": "user", "content": message}]

# 显示结构，确认无误
messages


[{'role': 'user',
  'content': 'Hello, Groq! This is my first ever message to you! Hi!'}]

In [4]:
# ========== 创建 Groq 客户端并做第一次 chat.completions 调用 ==========

# 实例化：默认从环境变量读 GROQ_API_KEY
groqcall = Groq()

# 非流式补全；model id 与 reasoning_effort 保留原样（影响路由与推理强度）
response = groqcall.chat.completions.create(model="openai/gpt-oss-20b", 
                                            messages=[{"role":"user", "content": "what is 2+2?"}],
                                            reasoning_effort="medium")

# 打印助手回复正文
print(response.choices[0].message.content)


2 + 2 = 4.


In [5]:
# ========== 试用抓取工具：拿 Gradio 官网正文 ==========

# 抓取 https://www.gradio.app 的清洗后文本
gradio = fetch_website_contents("https://www.gradio.app")
# 打印正文（注释写 first 500 characters，实际是整段 print；保持原逻辑）
print(gradio)  # print first 500 characters


Gradio

New
Gradio 6 is here!
Learn more
Update
Hackathon deadline approaching
Submit now
Build machine learning apps in Python
Create web interfaces for your ML models in minutes. Deploy anywhere,
			share with anyone.
Get Started
GitHub
40752
Click Me
Button
0
5
10
15
Plot
A
B
C
···
···
···
···
···
···
···
···
···
···
···
···
Dataframe
◢
◢
ImageSlider
0
100
Slider
Gallery
Accept terms
Checkbox
1
2
3
def
hello
():
print
(
"Hi"
)
return
42
Code
Click Me
Button
0
5
10
15
Plot
A
B
C
···
···
···
···
···
···
···
···
···
···
···
···
Dataframe
◢
◢
ImageSlider
0
100
Slider
Gallery
Accept terms
Checkbox
1
2
3
def
hello
():
print
(
"Hi"
)
return
42
Code
Hi! How can I help?
Hello!
👋
Chatbot
Good
sentiment
bad
HighlightedText
Model3D
Option 1
Option 2
Radio
Number
📁
Documents
📄
file.txt
📁
Images
FileExplorer
⋮⋮
Item 1
⋮⋮
Item 2
⋮⋮
Item 3
Draggable
▶
0:15
Audio
Face
Ear
AnnotatedImage
Option A
Dropdown
DateTime
Hi! How can I help?
Hello!
👋
Chatbot
Good
sentiment
bad
HighlightedText
Model3D
Option 

In [6]:
# ========== 定义 system 提示：吐槽风网页摘要助手 ==========
# 可自行实验，例如改成西班牙语 Markdown；正文保留英文

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [7]:
# ========== 定义 user 提示前缀：后面拼网页正文 ==========

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


In [8]:
# ========== 再练一次 messages 结构 + 带 reasoning_effort 的调用 ==========

# system 定角色，user 问简单问题（字符串保留英文）
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 与 OpenAI SDK 几乎同形的 chat.completions.create
response = groqcall.chat.completions.create(model="openai/gpt-oss-20b", 
                                            messages=messages,
                                            reasoning_effort="medium")


In [9]:
# ========== 打印上一格 response 的助手正文 ==========
print(response.choices[0].message.content)


4


In [10]:
# ========== 构造摘要任务的 messages：system + user(前缀+网页) ==========

# website：已抓取的网页正文
def messages_for(website):
    # 返回完整 messages 列表，稍后直接传给 chat.completions.create
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [11]:
# ========== 试一下：用 Gradio 正文看拼好的 messages ==========
# 可再换几个网站的抓取结果传入

messages_for(gradio)


[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nGradio\n\nNew\nGradio 6 is here!\nLearn more\nUpdate\nHackathon deadline approaching\nSubmit now\nBuild machine learning apps in Python\nCreate web interfaces for your ML models in minutes. Deploy anywhere,\n\t\t\tshare with anyone.\nGet Started\nGitHub\n40752\nClick Me\nButton\n0\n5\n10\n15\nPlot\nA\nB\nC\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\nDataframe\n◢\n◢\nImageSlider\n0\n100\nSlider\nGallery\nAccept terms\nCheckbox\n1\n2\n3\ndef\nhello\n():\nprint\n(\n"Hi"\n)\nreturn\n42\nCode\nClick 

In [12]:
# ========== 端到端：URL → 抓取 → Groq 摘要 ==========
# 你会很快熟悉这种封装！

def summarize(url):
    # 1) 抓网页
    website = fetch_website_contents(url)
    # 2) 调用 Groq chat.completions，messages 由 messages_for 生成
    response = groqcall.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages_for(website),
        reasoning_effort="medium"
        )
    # 3) 返回助手文本
    return response.choices[0].message.content


In [13]:
# ========== 直接调用 summarize，看原始摘要字符串 ==========
summarize("https://www.gradio.app")


'## Gradio: Because Your ML Model Deserves a Fancy Home\n\n- **New release alert** – Gradio\u202f6 just landed, so you can finally brag about “the newest” in your inbox.  \n- **Hackathon deadline** – There’s a looming submission deadline, so grab a coffee, code fast, and hope the judges don’t mind your caffeine‑induced bugs.  \n- **What it does** – Turns any Python ML model into a slick web app with zero JavaScript. One pip command, a single script, and boom: `http://127.0.0.1:7860`.  \n- **Components galore** – Images, audio, 3D, dataframes, even a tiny chatbot that probably knows nothing about your data.  \n- **Hosting** – “Permanent hosting” on Hugging Face Spaces for free, because we all love paying for bandwidth we never use.  \n- **Sharing** – Get a public link in seconds—perfect for impressing your boss who probably doesn’t care about your model.  \n\nBottom line: Gradio is the lazy developer’s dream—just paste your code, deploy, and pretend you actually built the UI.'

In [14]:
# ========== 用 Markdown 在输出区漂亮展示摘要 ==========

def display_summary(url):
    # 先拿到摘要
    summary = summarize(url)
    # 再渲染为 Markdown
    display(Markdown(summary))


In [15]:
# ========== 一键展示 Gradio 官网摘要 ==========
display_summary("https://www.gradio.app")


# Gradio: The “Make Your ML Models Public” Fandom

- **Gradio 6 just landed** – because apparently version 5 was *so* last year.  
- **Hackathon deadline is looming** – submit your project before your coffee runs out.  
- **One‑click Python magic** – install with `pip install gradio`, run `python app.py`, and voilà, a shiny web UI appears at `127.0.0.1:7860`.  
- **Component buffet** – images, audio, video, 3D, plots, dataframes… basically anything you can put on a screen.  
- **Free hosting on Hugging Face Spaces** – your demo stays online forever, auto‑scales, and comes with a URL that looks like a meme.  

Bottom line: If you want to turn your ML code into a pretty front‑end without learning any CSS, Gradio is the *lazy coder’s* dream.

In [ ]:
# ========== 占位空单元格 ==========
# 原笔记本此处为空；可忽略
